### [  모델 성능 향상을 위한 모델 구성 ] 

- 학습/연산 관련 층 
    * nn.Linear()  - 전결합층 FC Layer.  가중합 연산 수행

- 학습 안정화를 위한 층
    * nn.BatchNorm() 
        - 전층으로부터 전달 받은 값을 평균 0, 분산 1되도록 정리. AF 전에 진행
        - 배치 크기 만큼의 데이터에서 컬럼별 평균, 분산 계산
        - 학습모드 : 현재 배치의 컬럼별 평균, 분산 계산
        - 검증몯, : 학습 모드에서 설정된 평균, 분산 값을 적용

>> **[1] 모듈 로딩** 

In [1]:
## 모듈 로딩
import torch
import torch.nn as nn
import torch.nn.functional as F

## 재현성 설정 : 전역 난수 시드 
torch.manual_seed(0)

>> **[1] 일반 DNN 모델**

In [2]:
# ---------------------------------------------------------
# BatchNorm 없는 버전 (원본 MNISTDNN)
# ---------------------------------------------------------
class MNISTDNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.hd_layer  = nn.Linear(10, 7)
        self.dropout   = nn.Dropout()
        self.out_layer = nn.Linear(7, 4)

    def forward(self, data, verbose=False):
        weightedsum = self.hd_layer(data)
        if verbose: self._print_verbose(weightedsum, "Linear 출력")
        af_out = F.relu(weightedsum)
        out = self.dropout(af_out)
        return self.out_layer(out)

    def _print_verbose(self, resultTN, title):
        # detach() : 복사본 텐서를 만들고 rqquires_grad=False 설정
        #            업데이트 되지 않도록 설정
        mean_v = resultTN.detach().mean()
        std_v  = resultTN.detach().std()
        print(f"{title} 출력  : 평균 %{mean_v:7.4f}, 표준편차 %{std_v:7.4f} " )   

# ---------------------------------------------------------
# BatchNorm 추가 버전
# ---------------------------------------------------------
class MNISTDNN_BN(nn.Module):
    def __init__(self):
        super().__init__()
        self.hd_layer  = nn.Linear(10, 7)
        self.bn        = nn.BatchNorm1d(7)   ## 추가된 부분: Linear -> BN -> ReLU
        self.dropout   = nn.Dropout()
        self.out_layer = nn.Linear(7, 4)

    def forward(self, data, verbose=False):
        weightedsum = self.hd_layer(data)
        if verbose: self._print_verbose(weightedsum, "Linear 출력", "BN 적용 전")

        bn_out = self.bn(weightedsum)
        if verbose: self._print_verbose(bn_out, "BatchNorm 출력", "BN 적용 후")

        af_out = F.relu(bn_out)
        out = self.dropout(af_out)
        return self.out_layer(out)

    def _print_verbose(self, resultTN, title, kind):
        mean_v = resultTN.detach().mean()
        std_v  = resultTN.detach().std()
        print(f"{title:10} 출력  : 평균 %{mean_v:7.4f}, 표준편차 %{std_v:7.4f} ({kind})" )    

In [3]:
# ===========================================================
# 1) BatchNorm 적용 전/후 분포 비교
# ===========================================================
print("="*70)
print("1) BatchNorm 적용 전/후 분포 비교")
print("="*70)

# 일부러 스케일이 큰(평균 5, 표준편차 10 근처) 입력 -> 실제 raw feature 스케일 흉내
x = torch.randn(32, 10) * 10 + 5

print("\n[BatchNorm 모델] 배치 32개 forward")
model_bn = MNISTDNN_BN()
model_bn.train()
model_bn(x, verbose=True)

print("\n-> Linear 직후엔 평균/표준편차가 입력 스케일 그대로 크게 흔들리지만,")
print("   BatchNorm을 통과하면 배치 내에서 평균 0, 표준편차 1 근처로 강제 정규화됨")




1) BatchNorm 적용 전/후 분포 비교

[BatchNorm 모델] 배치 32개 forward
Linear 출력  출력  : 평균 % 3.2280, 표준편차 % 6.6274 (BN 적용 전)
BatchNorm 출력 출력  : 평균 % 0.0000, 표준편차 % 1.0022 (BN 적용 후)

-> Linear 직후엔 평균/표준편차가 입력 스케일 그대로 크게 흔들리지만,
   BatchNorm을 통과하면 배치 내에서 평균 0, 표준편차 1 근처로 강제 정규화됨


In [4]:
# ===========================================================
# 2) 학습 안정화 비교: BN 없는 모델 vs BN 있는 모델
# ===========================================================
print()
print("="*70)
print("2) 학습 안정화 비교 (일부러 큰 학습률 + 스케일 큰 입력 사용)")
print("="*70)

def make_data(n=64):
    X = torch.randn(n, 10) * 10 + 5     # 스케일이 크고 흔들리는 입력
    y = torch.randint(0, 4, (n,))
    return X, y

def train_and_log(model, lr=0.5, steps=30):
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    losses = []
    model.train()
    for step in range(steps):
        X, y = make_data()
        opt.zero_grad()
        logits = model(X)
        loss = loss_fn(logits, y)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses

torch.manual_seed(1)
model_plain = MNISTDNN()
losses_plain = train_and_log(model_plain)

torch.manual_seed(1)
model_bn2 = MNISTDNN_BN()
losses_bn = train_and_log(model_bn2)

import statistics as st
print("\n[BatchNorm 없음]")
print("  loss 처음 5개        :", [round(v,3) for v in losses_plain[:5]])
print("  loss 마지막 5개      :", [round(v,3) for v in losses_plain[-5:]])
print("  loss 표준편차 (변동성):", round(st.pstdev(losses_plain), 4))
print("  loss 최댓값          :", round(max(losses_plain), 4))

print("\n[BatchNorm 있음]")
print("  loss 처음 5개        :", [round(v,3) for v in losses_bn[:5]])
print("  loss 마지막 5개      :", [round(v,3) for v in losses_bn[-5:]])
print("  loss 표준편차 (변동성):", round(st.pstdev(losses_bn), 4))
print("  loss 최댓값          :", round(max(losses_bn), 4))


2) 학습 안정화 비교 (일부러 큰 학습률 + 스케일 큰 입력 사용)

[BatchNorm 없음]
  loss 처음 5개        : [3.557, 2.914, 2.17, 1.993, 1.85]
  loss 마지막 5개      : [1.662, 1.374, 1.372, 1.338, 1.533]
  loss 표준편차 (변동성): 0.5069
  loss 최댓값          : 3.5567

[BatchNorm 있음]
  loss 처음 5개        : [1.603, 1.475, 1.462, 1.468, 1.455]
  loss 마지막 5개      : [1.388, 1.413, 1.39, 1.378, 1.403]
  loss 표준편차 (변동성): 0.0455
  loss 최댓값          : 1.6025


In [5]:
torch.manual_seed(0)
model = MNISTDNN()
x = torch.rand(1, 10)
print("입력값:", x.round(decimals=2).tolist())

print("\n[train 모드] =====")
model.train()
for idx in range(3):
    print(f"\n--- {idx+1}번째 forward ---")
    model(x, verbose=True)

print("\n[eval 모드] =====") 
model.eval()
for idx in range(3):
    print(f"\n--- {idx+1}번째 forward ---")
    model(x, verbose=True)

입력값: [[0.5799999833106995, 0.30000001192092896, 0.800000011920929, 0.20000000298023224, 0.949999988079071, 0.8399999737739563, 0.07999999821186066, 0.3799999952316284, 0.5199999809265137, 0.5699999928474426]]

[train 모드] =====

--- 1번째 forward ---
Linear 출력 출력  : 평균 %-0.0686, 표준편차 % 0.4035 

--- 2번째 forward ---
Linear 출력 출력  : 평균 %-0.0686, 표준편차 % 0.4035 

--- 3번째 forward ---
Linear 출력 출력  : 평균 %-0.0686, 표준편차 % 0.4035 

[eval 모드] =====

--- 1번째 forward ---
Linear 출력 출력  : 평균 %-0.0686, 표준편차 % 0.4035 

--- 2번째 forward ---
Linear 출력 출력  : 평균 %-0.0686, 표준편차 % 0.4035 

--- 3번째 forward ---
Linear 출력 출력  : 평균 %-0.0686, 표준편차 % 0.4035 
